## 🎯 Learning Outcomes
### By the end of this module, you will be able to

- Understand **Series** and **DataFrame** — labeled 1-D and 2-D containers
- Perform **Indexing** with `.loc` (labels) and `.iloc` (positions)
- Handle **Inspection** and **missing-data**
- Perform **Type conversion**, **string operations**, and **sorting**
- Utilize **GroupBy**, **aggregation**, and **pivot tables**
- Apply **Merging**, **joining**, **concatenating**
- Use **`apply`**, **`map`**, and **method chaining** for clean pipelines
- Work with **Time series** and **categorical** dtypes

In [1]:
import numpy as np
import pandas as pd
print("Pandas:", pd.__version__)

Pandas: 2.2.2


## Part 1 — Series: a Labeled 1-D Array

A **Series** is a NumPy array with a labeled index. The index is what makes Pandas Pandas: arithmetic and joins **align by label**, not by position.

In [2]:
# -- Create a Series -----------------------------------------------------------
s = pd.Series([10, 20, 30, 40], index=['a', 'b', 'c', 'd'], name='counts')

print(s, "\n")

# -- Label-based access --------------------------------------------------------
print("s['b']    =", s['b'])
print("s[['a','c']] =", s[['a', 'c']].tolist())

# -- Vectorized operations -----------------------------------------------------
print("\ns * 2   :", (s * 2).tolist())
print("s.mean() :", s.mean())

a    10
b    20
c    30
d    40
Name: counts, dtype: int64 

s['b']    = 20
s[['a','c']] = [10, 30]

s * 2   : [20, 40, 60, 80]
s.mean() : 25.0


In [3]:
# -- Index alignment: the BIG difference from a list ---------------------------
s1 = pd.Series([1, 2, 3],   index=['a', 'b', 'c'])
s2 = pd.Series([10, 20, 30], index=['b', 'c', 'd'])

# Pandas aligns by LABEL, fills missing with NaN
print(s1 + s2)

a     NaN
b    12.0
c    23.0
d     NaN
dtype: float64


## Part 2 — DataFrame: Tabular Data

| Property | Meaning |
|---|---|
| `df.columns` | column labels |
| `df.index` | row labels |
| `df.shape` | `(rows, cols)` |
| `df['col']` | single column → Series |
| `df[['a', 'b']]` | multiple columns → DataFrame |

In [4]:
# -- Build from a dict of equal-length lists -----------------------------------
df = pd.DataFrame({
    'name':  ['Aisha', 'Omar', 'Lina', 'Tariq', 'Yara'],
    'age':   [22, 25, 23, 28, 21],
    'dept':  ['CS', 'EE', 'CS', 'ME', 'EE'],
    'score': [95.0, 88.5, 91.0, 76.5, 82.0],
})
df

,name,age,dept,score
0,Aisha,22,CS,95.0
1,Omar,25,EE,88.5
2,Lina,23,CS,91.0
3,Tariq,28,ME,76.5
4,Yara,21,EE,82.0


In [5]:
print("shape   :", df.shape)
print("columns :", list(df.columns))
print("dtypes  :")
print(df.dtypes)
print("\nname column (a Series):")
print(df['name'])

shape   : (5, 4)
columns : ['name', 'age', 'dept', 'score']
dtypes  :
name      object
age        int64
dept      object
score    float64
dtype: object

name column (a Series):
0    Aisha
1     Omar
2     Lina
3    Tariq
4     Yara
Name: name, dtype: object


In [6]:
# -- Read from common sources --------------------------------------------------
# (these are common patterns; we do NOT run them here)
# pd.read_csv('data.csv')
# pd.read_json('data.json')
# pd.read_excel('data.xlsx')
# pd.read_parquet('data.parquet')

# -- Build from a NumPy array --------------------------------------------------
arr = np.arange(12).reshape(4, 3)
pd.DataFrame(arr, columns=['x', 'y', 'z'])

,x,y,z
0,0,1,2
1,3,4,5
2,6,7,8
3,9,10,11


## Part 3 — Indexing: `.loc` vs `.iloc`

| Indexer | Selects by | Accepts |
|---|---|---|
| `.loc[r, c]` | **label** | labels, label-slices, boolean masks |
| `.iloc[r, c]` | **integer position** | ints, int-slices |

Use `.loc` for labels and conditions, `.iloc` for positions. Mixing them is the source of many beginner bugs.

In [7]:
# -- .loc: by label ------------------------------------------------------------
print("df.loc[0, 'name'] :", df.loc[0, 'name'])
print()
print("df.loc[0:2, ['name', 'age']]:")
print(df.loc[0:2, ['name', 'age']])
print()
print("df.loc[df['age'] > 23] -- boolean mask:")
print(df.loc[df['age'] > 23])

df.loc[0, 'name'] : Aisha

df.loc[0:2, ['name', 'age']]:
    name  age
0  Aisha   22
1   Omar   25
2   Lina   23

df.loc[df['age'] > 23] -- boolean mask:
    name  age dept  score
1   Omar   25   EE   88.5
3  Tariq   28   ME   76.5


In [8]:
# -- .iloc: by position --------------------------------------------------------
print("df.iloc[0, 0] :", df.iloc[0, 0])
print()
print("df.iloc[0:2, :] (first two rows):")
print(df.iloc[0:2, :])
print()
print("df.iloc[-1] (last row):")
print(df.iloc[-1])

df.iloc[0, 0] : Aisha

df.iloc[0:2, :] (first two rows):
    name  age dept  score
0  Aisha   22   CS   95.0
1   Omar   25   EE   88.5

df.iloc[-1] (last row):
name     Yara
age        21
dept       EE
score    82.0
Name: 4, dtype: object


In [9]:
# -- Setting values via .loc + condition ---------------------------------------
df2 = df.copy()
df2.loc[df2['score'] < 80, 'score'] = 80     # cap low scores at 80
df2

,name,age,dept,score
0,Aisha,22,CS,95.0
1,Omar,25,EE,88.5
2,Lina,23,CS,91.0
3,Tariq,28,ME,80.0
4,Yara,21,EE,82.0


## Part 4 — Inspecting Data

A first-look checklist before any analysis: shape, dtypes, missing values, distributions.

In [10]:
# -- First-look ----------------------------------------------------------------
print("shape:", df.shape)
print()
df.head(3)

shape: (5, 4)



,name,age,dept,score
0,Aisha,22,CS,95.0
1,Omar,25,EE,88.5
2,Lina,23,CS,91.0


In [11]:
df.describe()       # numeric summary

,age,score
count,5.000000,5.000000
mean,23.800000,86.600000
std,2.774887,7.360367
min,21.000000,76.500000
25%,22.000000,82.000000
50%,23.000000,88.500000
75%,25.000000,91.000000
max,28.000000,95.000000


In [12]:
df.describe(include='object')    # for string columns

,name,dept
count,5,5
unique,5,3
top,Aisha,CS
freq,1,2


In [13]:
# -- Frequencies ---------------------------------------------------------------
print("dept counts:")
print(df['dept'].value_counts())
print("\nunique depts :", df['dept'].nunique())

dept counts:
dept
CS    2
EE    2
ME    1
Name: count, dtype: int64

unique depts : 3


## Part 5 — Handling Missing Data

Pandas represents missing values with `NaN` (and `NaT` for datetimes). Strategies:

| Strategy | Method |
|---|---|
| Detect | `df.isna()`, `df.isna().sum()` |
| Drop | `df.dropna()`, `df.dropna(subset=[...])` |
| Impute (numeric) | `df['x'].fillna(df['x'].median())` |
| Impute (categorical) | `df['x'].fillna('unknown')` or the mode |
| Time series | `df.fillna(method='ffill')`, `df.interpolate()` |

In [14]:
# -- Inject some missing values ------------------------------------------------
dirty = df.copy()
dirty.loc[1, 'age']   = np.nan
dirty.loc[3, 'score'] = np.nan
dirty.loc[4, 'dept']  = np.nan
dirty

,name,age,dept,score
0,Aisha,22.0,CS,95.0
1,Omar,NaN,EE,88.5
2,Lina,23.0,CS,91.0
3,Tariq,28.0,ME,NaN
4,Yara,21.0,NaN,82.0


In [15]:
# -- Detection -----------------------------------------------------------------
print("missing per column:")
print(dirty.isna().sum())
print("\n% missing per column:")
print((dirty.isna().mean() * 100).round(1))

missing per column:
name     0
age      1
dept     1
score    1
dtype: int64

% missing per column:
name      0.0
age      20.0
dept     20.0
score    20.0
dtype: float64


In [16]:
# -- Imputation by column type --------------------------------------------------
clean = dirty.copy()
clean['age']   = clean['age'].fillna(clean['age'].median())          # robust
clean['score'] = clean['score'].fillna(clean['score'].mean())
clean['dept']  = clean['dept'].fillna('unknown')                     # category fallback
clean

,name,age,dept,score
0,Aisha,22.0,CS,95.000
1,Omar,22.5,EE,88.500
2,Lina,23.0,CS,91.000
3,Tariq,28.0,ME,89.125
4,Yara,21.0,unknown,82.000


## Part 6 — Type Conversion, Strings, and Sorting

In [17]:
# -- Type conversion -----------------------------------------------------------
df2 = df.copy()
df2['age']   = df2['age'].astype('int32')             # downcast
df2['score'] = pd.to_numeric(df2['score'])            # safe numeric parse
# For dates: pd.to_datetime('2026-05-09')
print(df2.dtypes)

name      object
age        int32
dept      object
score    float64
dtype: object


In [18]:
# -- The .str accessor: vectorized string operations ---------------------------
print(df['name'].str.lower().tolist())
print(df['name'].str.contains('a', case=False).tolist())
print(df['name'].str.len().tolist())
print(df['name'].str.replace('a', '@', regex=False).tolist())

['aisha', 'omar', 'lina', 'tariq', 'yara']
[True, True, True, True, True]
[5, 4, 4, 5, 4]
['Aish@', 'Om@r', 'Lin@', 'T@riq', 'Y@r@']


In [19]:
# -- Sorting -------------------------------------------------------------------
print("sort by score desc:")
print(df.sort_values('score', ascending=False))

print("\nsort by (dept asc, score desc):")
print(df.sort_values(['dept', 'score'], ascending=[True, False]))

print("\ntop 3 by score:")
print(df.nlargest(3, 'score'))

sort by score desc:
    name  age dept  score
0  Aisha   22   CS   95.0
2   Lina   23   CS   91.0
1   Omar   25   EE   88.5
4   Yara   21   EE   82.0
3  Tariq   28   ME   76.5

sort by (dept asc, score desc):
    name  age dept  score
0  Aisha   22   CS   95.0
2   Lina   23   CS   91.0
1   Omar   25   EE   88.5
4   Yara   21   EE   82.0
3  Tariq   28   ME   76.5

top 3 by score:
    name  age dept  score
0  Aisha   22   CS   95.0
2   Lina   23   CS   91.0
1   Omar   25   EE   88.5


## Part 7 — GroupBy and Aggregation

The split-apply-combine pattern: split the table by a key, apply a function, combine the results.

In [20]:
# -- Single-column aggregation -------------------------------------------------
print(df.groupby('dept')['score'].mean())

dept
CS    93.00
EE    85.25
ME    76.50
Name: score, dtype: float64


In [21]:
# -- Multiple aggregations at once ---------------------------------------------
df.groupby('dept').agg(
    avg_score = ('score', 'mean'),
    max_score = ('score', 'max'),
    n_students = ('score', 'count'),
)

,avg_score,max_score,n_students
dept,,,
CS,93.00,95.0,2
EE,85.25,88.5,2
ME,76.50,76.5,1


In [22]:
# -- transform: returns same shape as input (great for within-group z-scores) --
df_z = df.copy()
df_z['z_score'] = df_z.groupby('dept')['score'].transform(
    lambda s: (s - s.mean()) / s.std() if s.std() else 0.0
)
df_z

,name,age,dept,score,z_score
0,Aisha,22,CS,95.0,0.707107
1,Omar,25,EE,88.5,0.707107
2,Lina,23,CS,91.0,-0.707107
3,Tariq,28,ME,76.5,NaN
4,Yara,21,EE,82.0,-0.707107


In [23]:
# -- Pivot table for cross-tabulation ------------------------------------------
extra = df.assign(year=[1, 2, 1, 2, 1])
pd.pivot_table(extra, values='score', index='dept', columns='year', aggfunc='mean')

year,1,2
dept,,
CS,93.0,NaN
EE,82.0,88.5
ME,NaN,76.5


## Part 8 — Merging and Joining

| `how=` | Result |
|---|---|
| `'inner'` | only rows present in **both** tables |
| `'left'` | every row in the **left** table |
| `'right'` | every row in the **right** table |
| `'outer'` | every row in **either** table |

In [24]:
students = pd.DataFrame({'id': [1, 2, 3], 'name': ['A', 'B', 'C']})
grades   = pd.DataFrame({'id': [1, 2, 4], 'grade': [95, 88, 70]})

print("students:"); print(students)
print("\ngrades  :"); print(grades)

students:
   id name
0   1    A
1   2    B
2   3    C

grades  :
   id  grade
0   1     95
1   2     88
2   4     70


In [25]:
# -- Inner: only rows present in both ------------------------------------------
print("inner:")
print(pd.merge(students, grades, on='id', how='inner'))

# -- Left: every student, NaN if no grade --------------------------------------
print("\nleft:")
print(pd.merge(students, grades, on='id', how='left'))

# -- Outer: every row in either side -------------------------------------------
print("\nouter:")
print(pd.merge(students, grades, on='id', how='outer'))

inner:
   id name  grade
0   1    A     95
1   2    B     88

left:
   id name  grade
0   1    A   95.0
1   2    B   88.0
2   3    C    NaN

outer:
   id name  grade
0   1    A   95.0
1   2    B   88.0
2   3    C    NaN
3   4  NaN   70.0


In [26]:
# -- concat: stack rows or columns ---------------------------------------------
top    = df.head(2)
bottom = df.tail(2)
print("axis=0 (stack rows):")
print(pd.concat([top, bottom], axis=0))

axis=0 (stack rows):
    name  age dept  score
0  Aisha   22   CS   95.0
1   Omar   25   EE   88.5
3  Tariq   28   ME   76.5
4   Yara   21   EE   82.0


## Part 9 — `apply`, `map`, and Method Chaining

In [27]:
# -- map: element-wise on a Series ---------------------------------------------
df['score'].map(lambda x: x / 100).tolist()

[0.95, 0.885, 0.91, 0.765, 0.82]

In [28]:
# -- apply with axis=1: row-wise ----------------------------------------------
# (slow! prefer vectorized expressions when possible)
df.assign(card=df.apply(lambda r: f"{r['name']} ({r['age']})", axis=1))

,name,age,dept,score,card
0,Aisha,22,CS,95.0,Aisha (22)
1,Omar,25,EE,88.5,Omar (25)
2,Lina,23,CS,91.0,Lina (23)
3,Tariq,28,ME,76.5,Tariq (28)
4,Yara,21,EE,82.0,Yara (21)


In [29]:
# -- Method chaining: a clean preprocessing pipeline ---------------------------
clean_df = (
    df.copy()
      .assign(age_group = lambda d: pd.cut(d['age'], [0, 22, 25, 99]))
      .query('score >= 80')
      .sort_values('score', ascending=False)
      .reset_index(drop=True)
)
clean_df

,name,age,dept,score,age_group
0,Aisha,22,CS,95.0,"(0, 22]"
1,Lina,23,CS,91.0,"(22, 25]"
2,Omar,25,EE,88.5,"(22, 25]"
3,Yara,21,EE,82.0,"(0, 22]"


## Part 10 — Time Series and Categoricals

A `DatetimeIndex` unlocks `resample`, `rolling`, `shift`, and `diff`. The `category` dtype saves memory when string values repeat.

In [30]:
# -- Build a small time series -------------------------------------------------
dates = pd.date_range('2026-01-01', periods=30, freq='D')
ts = pd.DataFrame({
    'date':  dates,
    'price': 100 + np.cumsum(np.random.default_rng(0).normal(0, 1, 30)),
}).set_index('date')

ts.head()

,price
date,
2026-01-01,100.125730
2026-01-02,99.993625
2026-01-03,100.634048
2026-01-04,100.738948
2026-01-05,100.203279


In [31]:
# -- Resample: change the frequency --------------------------------------------
print("Weekly mean:")
print(ts.resample('W').mean().round(3))

Weekly mean:
              price
date               
2026-01-04  100.373
2026-01-11  101.234
2026-01-18   96.889
2026-01-25   96.821
2026-02-01   96.971


In [32]:
# -- Rolling window + period-over-period change --------------------------------
ts['rolling7']  = ts['price'].rolling(7).mean()
ts['change']    = ts['price'].diff()
ts.tail()

,price,rolling7,change
date,,,
2026-01-26,98.258160,97.244407,0.094012
2026-01-27,97.514661,97.412725,-0.743499
2026-01-28,96.592936,97.467730,-0.921725
2026-01-29,96.135210,97.262137,-0.457726
2026-01-30,96.355405,97.183028,0.220195


In [33]:
# -- Categorical dtype: memory savings on repeated values ----------------------
df_cat = df.copy()
print("object dtype memory:", df_cat['dept'].memory_usage(deep=True), "bytes")

df_cat['dept'] = df_cat['dept'].astype('category')
print("category memory   :", df_cat['dept'].memory_usage(deep=True), "bytes")
print("\ncategories      :", df_cat['dept'].cat.categories.tolist())

object dtype memory: 387 bytes
category memory   : 398 bytes

categories      : ['CS', 'EE', 'ME']


## Part 11 — Reading a CSV File with Pandas

In practice, data almost always comes from a **CSV file**. `pd.read_csv()` is the single most-used Pandas function — it returns a fully formed DataFrame in one line, with automatic dtype inference, header detection, and dozens of options for messy files.

| Parameter | Purpose |
|---|---|
| `filepath` | path (or URL) to the CSV |
| `dtype=` | override column types |
| `usecols=` | load only specific columns |
| `parse_dates=` | parse date columns automatically |
| `na_values=` | extra strings to treat as NaN |

We will work with **`students_scores.csv`** — 15 students with Math, Science, and English exam scores plus attendance and a pass/fail flag.

In [34]:
import pandas as pd
import numpy as np

# -- Load the CSV — one line gives us a complete DataFrame -----------------
df = pd.read_csv('students_scores.csv')

# -- First-look checklist --------------------------------------------------
print('Shape         :', df.shape)          # (rows, cols)
print('Columns       :', df.columns.tolist())
print()
print('Dtypes:')
print(df.dtypes)
print()
df.head()

Shape         : (15, 8)
Columns       : ['student_id', 'name', 'age', 'math_score', 'science_score', 'english_score', 'attendance_pct', 'passed']

Dtypes:
student_id          int64
name               object
age                 int64
math_score          int64
science_score       int64
english_score       int64
attendance_pct    float64
passed               bool
dtype: object



,student_id,name,age,math_score,science_score,english_score,attendance_pct,passed
0,1,Ali Hassan,18,88,92,75,96.5,True
1,2,Sara Al-Mutairi,19,72,68,84,88.0,True
2,3,Khalid Nasser,18,55,60,58,74.5,False
3,4,Fatima Al-Zahra,20,95,97,91,99.0,True
4,5,Omar Bin Saleh,19,43,50,62,65.0,False


In [35]:
# -- Statistical snapshot of every numeric column -------------------------
df.describe().round(2)

,student_id,age,math_score,science_score,english_score,attendance_pct
count,15.00,15.00,15.00,15.00,15.00,15.00
mean,8.00,19.13,71.07,73.20,73.13,84.43
std,4.47,1.06,18.65,16.59,15.97,13.92
min,1.00,18.00,38.00,45.00,44.00,58.00
25%,4.50,18.00,57.50,62.50,60.00,76.75
50%,8.00,19.00,76.00,74.00,77.00,89.50
75%,11.50,20.00,86.50,85.50,86.00,95.50
max,15.00,21.00,95.00,97.00,93.00,99.00


In [36]:
# -- Check for missing values before doing anything else ------------------
print('Missing values per column:')
print(df.isna().sum())
print()
print('Pass / Fail breakdown:')
print(df['passed'].value_counts())

Missing values per column:
student_id        0
name              0
age               0
math_score        0
science_score     0
english_score     0
attendance_pct    0
passed            0
dtype: int64

Pass / Fail breakdown:
passed
True     11
False     4
Name: count, dtype: int64


### Selecting and filtering

All the `.loc` / `.iloc` and boolean-mask techniques you learned in Parts 3–5 work directly on data loaded from a CSV.

In [37]:
# -- Select specific columns ----------------------------------------------
scores_only = df[['name', 'math_score', 'science_score', 'english_score']]
print('Score columns (first 5):')
print(scores_only.head())

# -- Filter: students who failed ------------------------------------------
failed = df[df['passed'] == False]
print(f'\nFailed students ({len(failed)}):')
print(failed[['name', 'math_score', 'science_score', 'english_score', 'attendance_pct']])

Score columns (first 5):
              name  math_score  science_score  english_score
0       Ali Hassan          88             92             75
1  Sara Al-Mutairi          72             68             84
2    Khalid Nasser          55             60             58
3  Fatima Al-Zahra          95             97             91
4   Omar Bin Saleh          43             50             62

Failed students (4):
              name  math_score  science_score  english_score  attendance_pct
2    Khalid Nasser          55             60             58            74.5
4   Omar Bin Saleh          43             50             62            65.0
8   Tariq Al-Jabri          38             45             49            58.0
12  Saad Al-Otaibi          47             52             44            61.0


In [38]:
# -- Multi-condition filter: low attendance AND failing score in any subject
at_risk = df[
    (df['attendance_pct'] < 75) |
    (df['math_score']    < 50) |
    (df['science_score'] < 50)
]
print(f'At-risk students ({len(at_risk)}):')
print(at_risk[['name', 'math_score', 'science_score', 'attendance_pct', 'passed']])

At-risk students (4):
              name  math_score  science_score  attendance_pct  passed
2    Khalid Nasser          55             60            74.5   False
4   Omar Bin Saleh          43             50            65.0   False
8   Tariq Al-Jabri          38             45            58.0   False
12  Saad Al-Otaibi          47             52            61.0   False


### Adding computed columns

Use `.assign()` for a clean, chainable way to add derived columns without modifying the original DataFrame.

In [39]:
# -- Weighted average: Math 40 %, Science 35 %, English 25 % -------------
df = df.assign(
    weighted_avg = lambda d: (
        d['math_score']    * 0.40 +
        d['science_score'] * 0.35 +
        d['english_score'] * 0.25
    ).round(1),
    grade = lambda d: pd.cut(
        d['math_score'],
        bins  = [0, 49, 59, 69, 79, 100],
        labels= ['F', 'D', 'C', 'B', 'A']
    )
)
df[['name', 'math_score', 'science_score', 'english_score',
    'weighted_avg', 'grade']].sort_values('weighted_avg', ascending=False)

,name,math_score,science_score,english_score,weighted_avg,grade
3,Fatima Al-Zahra,95,97,91,94.7,A
11,Lina Mahmoud,91,95,89,91.9,A
7,Maryam Ibrahim,90,88,93,90.0,A
0,Ali Hassan,88,92,75,86.2,A
9,Hessa Al-Mansoori,82,79,88,82.4,A
14,Ziad Al-Harbi,85,83,77,82.3,A
13,Reem Al-Hamdan,76,80,82,78.9,B
5,Noura Al-Rashidi,78,74,80,77.1,B
1,Sara Al-Mutairi,72,68,84,73.6,B
6,Youssef Khalil,66,70,55,64.6,C


### GroupBy aggregation on CSV data

The split-apply-combine pattern from Part 7 is especially powerful once you have real data. Here we group by the letter grade and compare subject averages.

In [40]:
# -- Average scores and attendance per grade bucket -----------------------
summary = (
    df.groupby('grade', observed=True)
      .agg(
          n_students    = ('name',           'count'),
          avg_math      = ('math_score',     'mean'),
          avg_science   = ('science_score',  'mean'),
          avg_english   = ('english_score',  'mean'),
          avg_attend    = ('attendance_pct', 'mean'),
      )
      .round(1)
)
print(summary)


       n_students  avg_math  avg_science  avg_english  avg_attend
grade                                                            
F               3      42.7         49.0         51.7        61.3
D               1      55.0         60.0         58.0        74.5
C               2      63.0         67.5         62.5        80.8
B               3      75.3         74.0         82.0        89.5
A               6      88.5         89.0         85.5        96.3


In [41]:
# -- Top 5 students by weighted average ------------------------------------
print('Top 5 students:')
print(
    df.nlargest(5, 'weighted_avg')
      [['name', 'weighted_avg', 'grade', 'attendance_pct']]
      .to_string(index=False)
)

# -- Pass rate -----------------------------------------------------------------
pass_rate = df['passed'].mean() * 100
print(f'\nOverall pass rate: {pass_rate:.0f}%')

Top 5 students:
             name  weighted_avg grade  attendance_pct
  Fatima Al-Zahra          94.7     A            99.0
     Lina Mahmoud          91.9     A            98.0
   Maryam Ibrahim          90.0     A            97.0
       Ali Hassan          86.2     A            96.5
Hessa Al-Mansoori          82.4     A            94.5

Overall pass rate: 73%


In [42]:
# -- Save the enriched DataFrame back to CSV --------------------------------
df.to_csv('students_scores_enriched.csv', index=False)
print('Saved students_scores_enriched.csv')
print(pd.read_csv('students_scores_enriched.csv').columns.tolist())

Saved students_scores_enriched.csv
['student_id', 'name', 'age', 'math_score', 'science_score', 'english_score', 'attendance_pct', 'passed', 'weighted_avg', 'grade']


> **Key takeaways — CSV + Pandas**
>
> | Step | Code |
> |---|---|
> | Load | `df = pd.read_csv('file.csv')` |
> | First look | `df.head()`, `df.dtypes`, `df.describe()`, `df.isna().sum()` |
> | Select columns | `df[['col1', 'col2']]` |
> | Filter rows | `df[df['x'] > threshold]`, multi-condition with `&` / `\|` |
> | Add columns | `df.assign(new_col = lambda d: ...)` |
> | Bin numerics | `pd.cut(series, bins=[...], labels=[...])` |
> | Aggregate | `df.groupby('col').agg(name=('col', func))` |
> | Top-N rows | `df.nlargest(N, 'col')` |
> | Save | `df.to_csv('output.csv', index=False)` |
>
> Compared with the NumPy approach from the previous module, Pandas gives you **named columns**, **built-in missing-value handling**, and expressive methods like `groupby` and `pd.cut` — all without writing a single loop.

---

## Worked Examples

### Example 1 — Sales analysis pipeline

Take a raw transactions table, clean it, group by region and product, rank top categories, and compute month-over-month growth — all in one chained pipeline.

In [43]:
rng = np.random.default_rng(7)

# -- Synthesize raw transactions -----------------------------------------------
n = 500
sales = pd.DataFrame({
    'date':     pd.to_datetime('2026-01-01') + pd.to_timedelta(rng.integers(0, 90, n), unit='D'),
    'region':   rng.choice(['East', 'West', 'North', 'South'], n),
    'product':  rng.choice(['A', 'B', 'C', 'D'], n, p=[0.4, 0.3, 0.2, 0.1]),
    'quantity': rng.integers(1, 10, n),
    'price':    rng.uniform(5, 100, n).round(2),
})
# Inject some dirt
sales.loc[sales.sample(20, random_state=0).index, 'price'] = np.nan
sales.head()

,date,region,product,quantity,price
0,2026-03-27,West,B,6,90.26
1,2026-02-26,West,C,8,16.71
2,2026-03-03,North,C,3,55.66
3,2026-03-22,North,B,8,9.75
4,2026-02-22,West,C,1,94.22


In [44]:
# -- Clean + enrich + aggregate in one chain -----------------------------------
report = (
    sales
      .dropna(subset=['price'])                           # drop bad rows
      .assign(revenue = lambda d: d['quantity'] * d['price'],
              month   = lambda d: d['date'].dt.to_period('M').astype(str))
      .groupby(['region', 'product'], as_index=False)
      .agg(total_revenue = ('revenue', 'sum'),
           n_orders      = ('revenue', 'count'),
           avg_ticket    = ('revenue', 'mean'))
      .sort_values('total_revenue', ascending=False)
)
report.head(10).round(2)

,region,product,total_revenue,n_orders,avg_ticket
4,North,A,15843.50,44,360.08
8,South,A,15500.22,57,271.93
0,East,A,15003.96,46,326.17
12,West,A,13198.59,55,239.97
9,South,B,9717.90,41,237.02
1,East,B,8954.05,34,263.35
10,South,C,8269.36,32,258.42
5,North,B,7005.43,33,212.29
13,West,B,6771.57,32,211.61
2,East,C,6152.26,24,256.34


In [45]:
# -- Top-3 products per region (uses groupby + nlargest) -----------------------
top3 = (
    report
      .groupby('region', group_keys=False)
      .apply(lambda g: g.nlargest(3, 'total_revenue'))
      .reset_index(drop=True)
)
top3.round(2)

/tmp/ipykernel_2619/1464784027.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.nlargest(3, 'total_revenue'))


,region,product,total_revenue,n_orders,avg_ticket
0,East,A,15003.96,46,326.17
1,East,B,8954.05,34,263.35
2,East,C,6152.26,24,256.34
3,North,A,15843.50,44,360.08
4,North,B,7005.43,33,212.29
5,North,C,5229.66,17,307.63
6,South,A,15500.22,57,271.93
7,South,B,9717.90,41,237.02
8,South,C,8269.36,32,258.42
9,West,A,13198.59,55,239.97


### Example 2 — Customer RFM scoring

Recency-Frequency-Monetary scoring: bucket each metric into quartiles 1–4 and combine into a single score. A classic marketing-analytics workflow.

In [46]:
rng = np.random.default_rng(11)

# -- Synthesize a customer-level summary table ---------------------------------
customers = pd.DataFrame({
    'customer_id': range(1, 21),
    'last_order_days_ago': rng.integers(1, 365, 20),
    'orders_count':        rng.integers(1, 50, 20),
    'total_spend':         rng.uniform(50, 5000, 20).round(2),
})
customers.head()

,customer_id,last_order_days_ago,orders_count,total_spend
0,1,49,43,4905.52
1,2,47,19,1062.32
2,3,291,8,2790.97
3,4,182,26,2443.94
4,5,215,22,1798.71


In [47]:
# -- Score each metric on a 1-4 scale (qcut = quantile-based binning) ----------
# Recency: SMALLER is better -> reverse the labels
customers['R'] = pd.qcut(customers['last_order_days_ago'], 4, labels=[4, 3, 2, 1]).astype(int)
customers['F'] = pd.qcut(customers['orders_count'],         4, labels=[1, 2, 3, 4]).astype(int)
customers['M'] = pd.qcut(customers['total_spend'],          4, labels=[1, 2, 3, 4]).astype(int)

customers['RFM']     = customers['R'] + customers['F'] + customers['M']
customers['segment'] = pd.cut(customers['RFM'],
                              bins=[0, 5, 8, 12],
                              labels=['Low', 'Mid', 'High'])

customers.sort_values('RFM', ascending=False).head(10)

,customer_id,last_order_days_ago,orders_count,total_spend,R,F,M,RFM,segment
0,1,49,43,4905.52,4,4,4,12,High
8,9,177,42,4343.30,3,4,4,11,High
13,14,26,33,4484.92,4,3,4,11,High
18,19,357,42,4512.08,1,4,4,9,High
7,8,11,14,4020.90,4,1,4,9,High
6,7,260,49,1214.74,2,4,2,8,Mid
3,4,182,26,2443.94,3,2,3,8,Mid
16,17,275,47,3383.14,1,4,3,8,Mid
5,6,219,33,2978.40,2,3,3,8,Mid
1,2,47,19,1062.32,4,2,1,7,Mid


### Example 3 — Joining a dimension table to a fact table

A common warehouse pattern: fact table has IDs only; dimension table has the descriptions. Left-merge to enrich.

In [48]:
# -- Fact table: thin, just IDs and metrics ------------------------------------
orders = pd.DataFrame({
    'order_id':     [101, 102, 103, 104, 105],
    'product_id':   [1, 2, 1, 3, 2],
    'customer_id':  ['c1', 'c2', 'c2', 'c1', 'c3'],
    'amount':       [25.0, 17.5, 25.0, 99.9, 17.5],
})

# -- Dimension table: rich descriptions ----------------------------------------
products = pd.DataFrame({
    'product_id': [1, 2, 3, 4],
    'name':       ['Notebook', 'Pen', 'Calculator', 'Marker'],
    'category':   ['Stationery', 'Stationery', 'Electronics', 'Stationery'],
})

# -- Left merge enriches every order with product info -------------------------
enriched = orders.merge(products, on='product_id', how='left')
enriched

,order_id,product_id,customer_id,amount,name,category
0,101,1,c1,25.0,Notebook,Stationery
1,102,2,c2,17.5,Pen,Stationery
2,103,1,c2,25.0,Notebook,Stationery
3,104,3,c1,99.9,Calculator,Electronics
4,105,2,c3,17.5,Pen,Stationery


In [49]:
# -- Aggregate the enriched table ---------------------------------------------
enriched.groupby('category').agg(
    total_amount = ('amount', 'sum'),
    n_orders     = ('order_id', 'count'),
).round(2)

,total_amount,n_orders
category,,
Electronics,99.9,1
Stationery,85.0,4


---

## Summary

| Concept | Key Point |
|---|---|
| **Series** | Labeled 1-D array; aligns by label |
| **DataFrame** | Tabular data with labeled rows/columns |
| **`.loc`** | Indexing by **label** or boolean mask |
| **`.iloc`** | Indexing by **integer position** |
| **Inspection** | `df.head()`, `df.info()`, `df.describe()`, `value_counts()` for initial overview |
| **Missing Data** | `df.isna()`, `dropna()`, `fillna()` to handle `NaN` values |
| **Type Conversion** | `astype()`, `pd.to_numeric()`, `pd.to_datetime()` |
| **String Ops** | `.str` accessor for vectorized string methods |
| **Sorting** | `sort_values()`, `nlargest()` |
| **GroupBy** | Split-apply-combine pattern for aggregation (`.agg()`) or transformation (`.transform()`) |
| **Pivot Tables** | Reshape data for cross-tabulation with `pd.pivot_table()` |
| **Merging** | `pd.merge()` for combining DataFrames based on keys (`inner`, `left`, `right`, `outer`) |
| **Concatenating** | `pd.concat()` for stacking rows or columns |
| **`apply` & `map`** | Element-wise (`.map()`) or row/column-wise (`.apply()`) operations |
| **Method Chaining** | Clean pipelines for data preprocessing |
| **Time Series** | `DatetimeIndex` enables `resample()`, `rolling()`, `diff()` |
| **Categoricals** | `category` dtype for memory-efficient storage of repeated string values |

### Contributed by: Abdulhadi Zubailah